# Ngày 20 · Kỹ thuật đặc trưng và target encoding

- Mã hóa cột phân loại nhiều giá trị bằng trung bình mục tiêu có làm mượt
- Thấy rò rỉ khi mã hóa trên chính các mẫu dùng để huấn luyện
- Mã hóa out-of-fold để mỗi mẫu không thấy nhãn của chính nó

**Đọc:**
- Micci-Barreca (2001), A Preprocessing Scheme for High-Cardinality Categorical Attributes in Classification and Prediction Problems, SIGKDD Explorations 3(1), 27–32
- Kuhn & Johnson (2019), Feature Engineering and Selection, Chương 5 Encoding Categorical Predictors
- ISLR (bản 2), §3.3.1 Qualitative Predictors (biến giả)

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.metrics import roc_auc_score
def kfold_indices(n, k, seed):
    parts = np.array_split(np.random.default_rng(seed).permutation(n), k)
    return [(np.concatenate(parts[:i] + parts[i + 1:]), parts[i]) for i in range(k)]
_r20 = np.random.default_rng(20)
_ids = np.array([f'id{i}' for i in _r20.integers(0, 300, 1000)]); _ynoise = _r20.integers(0, 2, 1000)   # giả lập: nhãn không liên quan
_cats = _r20.choice(list('ABCDEFGHIJ'), 1000)
_ysig = (_r20.random(1000) < 1 / (1 + np.exp(-np.vectorize(dict(zip('ABCDEFGHIJ', np.linspace(-1.5, 1.5, 10))).get)(_cats)))).astype(int)
def _ref_fit(cat, y, m):
    s = pd.Series(np.asarray(y, float)); g = s.groupby(pd.Series(cat)).agg(['sum', 'count']); prior = float(s.mean())
    return {'map': ((g['sum'] + m * prior) / (g['count'] + m)).to_dict(), 'prior': prior}
def _ref_oof(cat, y, k, seed, m):
    cat, y, out = np.asarray(cat, dtype=object), np.asarray(y, float), np.empty(len(y))
    for tr, va in kfold_indices(len(y), k, seed):
        e = _ref_fit(cat[tr], y[tr], m); out[va] = pd.Series(cat[va]).map(e['map']).fillna(e['prior']).to_numpy(float)
    return out


## Target encoding có làm mượt và out-of-fold

Gợi ý: Dùng pandas: pd.Series(y).groupby(pd.Series(cat)).agg(["sum", "count"]). Khi apply, .map(...) rồi .fillna(prior). Out-of-fold: lặp kfold_indices (đã có sẵn), fit trên train của fold, apply lên phần validation của fold.

In [ ]:
def target_encode_fit(cat, y, m):
    # Trả {'map': {giá trị: mã}, 'prior': trung bình y}. mã = (Σy + m·prior) / (số mẫu + m).
    raise NotImplementedError

def target_encode_apply(enc, cat):
    # Trả mảng float; giá trị chưa gặp nhận prior.
    raise NotImplementedError

def oof_target_encode(cat, y, k, seed, m):
    # Mỗi mẫu được mã hóa bằng thống kê từ k − 1 fold còn lại (kfold_indices(len(y), k, seed)).
    raise NotImplementedError


In [ ]:
_report('d20-target',
    _check('Ví dụ tính tay với m = 2', lambda: ((lambda e: np.isclose(e['prior'], 2 / 3) and np.allclose([e['map'][c] for c in 'abc'], [7 / 12, 2 / 3, 7 / 9]))(target_encode_fit(np.array(list('aabbbc')), np.array([1, 0, 1, 1, 0, 1]), 2)), 'a = 0,583; b = 0,667; c = 0,778.')),
    _check('Giá trị chưa gặp nhận prior', lambda: (np.allclose(target_encode_apply(target_encode_fit(np.array(list('aab')), np.array([1, 0, 1]), 1), np.array(['a', 'z'])), [(1 + 2 / 3) / 3, 2 / 3]), '"z" không có trong train.')),
    _check('Out-of-fold khớp cách tính tham chiếu', lambda: (np.allclose(oof_target_encode(_cats, _ysig, 5, 0, 10), _ref_oof(_cats, _ysig, 5, 0, 10)), '5 fold, seed 0, m = 10.')),
    _check('Không rò rỉ trên nhãn ngẫu nhiên, vẫn học được khi có tín hiệu', lambda: ((lambda a, b: a < .62 and b > .65)(roc_auc_score(_ynoise, oof_target_encode(_ids, _ynoise, 5, 0, 0)), roc_auc_score(_ysig, oof_target_encode(_cats, _ysig, 5, 0, 10))), 'Nhãn ngẫu nhiên: AUC phải gần 0,5 (bản trong-mẫu cho 0,82).')),
)


## Dự án

Trên House Prices, so ba cách cho cột Neighborhood (25 giá trị) khi dự đoán log(SalePrice) bằng Ridge: one-hot, target encoding trong-mẫu (sai) và target encoding out-of-fold (đúng), đánh giá bằng 5-fold. Thêm log1p cho các cột lệch (GrLivArea, LotArea) và ghi thay đổi RMSE của từng bước.